### Baseline Preditiva: Regressão na Base Integral

In [20]:
from pathlib import Path
import pandas as pd

base_path = Path.cwd().parent
file_path = base_path / "data" / "processed" / "Base_de_Dados_Processada.csv"

df = pd.read_csv(file_path)

df.head()

,População_2022,Pontuação Indice ODS 2025,Goal 1 Score,Goal 2 Score,Goal 3 Score,Goal 5 Score,Goal 6 Score,Goal 7 Score,Goal 8 Score,Goal 9 Score,...,Normalizado (0-100): SDG15_8_UNI_CNS,Normalizado (0-100): SDG15_PRTAMB,Normalizado (0-100): SDG16_1_OB_HM,Normalizado (0-100): SDG16_3_M_ARM_FG,Normalizado (0-100): SDG16_4_TX_HMC,Normalizado (0-100): SDG16_CRP,Normalizado (0-100): SDG16_DRTHMN,Normalizado (0-100): SDG16_TRANSP,Normalizado (0-100): SDG17_1_INVST_PB,Normalizado (0-100): SDG17_3_P_RC_TRB
0,21495.0,41.885064,53.769451,53.255278,66.451721,58.730307,20.177142,56.309444,46.940699,11.631238,...,30.659,80.0,0.0,0.0,38.784211,85.714,42.857,66.667,0.000000,8.108054
1,96833.0,45.475390,54.219225,50.890756,65.677811,41.654565,72.408554,68.007222,49.281375,14.603964,...,0.013,80.0,0.0,0.0,32.057895,85.714,57.143,66.667,1.133196,34.298246
2,5363.0,47.569518,49.228293,43.817398,89.494428,48.938353,44.025511,63.241944,54.613778,7.181381,...,0.000,0.0,0.0,0.0,1.642105,85.714,42.857,66.667,38.715329,8.963317
3,86895.0,48.582323,50.752090,51.083977,73.026011,47.023041,66.630268,69.962222,45.581682,17.049953,...,0.012,80.0,0.0,0.0,63.655263,85.714,57.143,50.000,12.187507,40.743620
4,15890.0,48.718897,56.552994,50.268036,79.958615,32.086768,65.794541,68.762222,52.890151,11.397874,...,40.055,80.0,100.0,0.0,33.755263,100.000,42.857,66.667,8.461212,21.905901


Importação das dependências e modelos 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Modelos
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

Separação das variáveis alvo

In [22]:
target_col_ideb_af = 'Normalizado (0-100): SDG4_15_IDEB_AF'
target_col_ideb_ai = 'Normalizado (0-100): SDG4_16_IDEB_AI'

targets = df[[target_col_ideb_af, target_col_ideb_ai]].copy()

Divisão da base de dados para treino e teste 

In [23]:

def avaliar_modelos_por_alvo(df_base, coluna_alvo, colunas_remover):
    print(f"\n--- Avaliação Baseline: {coluna_alvo} ---")
    
    df_clean = df_base.dropna(subset=[coluna_alvo])
    
    X = df_clean.drop(columns=colunas_remover)
    y = df_clean[coluna_alvo].values.ravel()
    
    # Divisão em Treino e Teste (80/20)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Padronização das features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    modelos = {
        'Random Forest': RandomForestRegressor(random_state=42),
        'XGBoost': XGBRegressor(random_state=42),
        'LightGBM': LGBMRegressor(random_state=42, verbose=-1),
        'CatBoost': CatBoostRegressor(random_state=42, verbose=0)
    }

    resultados = []
    for nome, modelo in modelos.items():
        modelo.fit(X_train_scaled, y_train)
        y_pred = modelo.predict(X_test_scaled)
        
        resultados.append({
            'Modelo': nome,
            'MAE': mean_absolute_error(y_test, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            'R²': r2_score(y_test, y_pred)
        })
        
    return pd.DataFrame(resultados).sort_values(by='R²', ascending=False)

Exibição dos resultados 

In [24]:

# Anos Finais
resultados_af = avaliar_modelos_por_alvo(df, target_col_ideb_af, targets)
display(resultados_af)

# Anos Iniciais
resultados_ai = avaliar_modelos_por_alvo(df, target_col_ideb_ai, targets)
display(resultados_ai)


--- Avaliação Baseline: Normalizado (0-100): SDG4_15_IDEB_AF ---


,Modelo,MAE,RMSE,R²
3,CatBoost,8.089062,10.831763,0.403994
2,LightGBM,8.038448,10.943785,0.391602
0,Random Forest,8.081688,10.999790,0.385359
1,XGBoost,8.582812,11.635110,0.312309



--- Avaliação Baseline: Normalizado (0-100): SDG4_16_IDEB_AI ---


,Modelo,MAE,RMSE,R²
3,CatBoost,8.384808,11.041930,0.621059
2,LightGBM,8.533707,11.232066,0.607897
0,Random Forest,8.624859,11.346493,0.599867
1,XGBoost,9.166807,11.960900,0.555360
